# SOME TITLE
---

## Introduction

- provide some relevant background information on the topic so that someone unfamiliar with it will be prepared to understand the rest of your report
- clearly state the question you tried to answer with your project
- identify and fully describe the dataset that was used to answer the question

## Methods and results

- You may include references if necessary, as long as they all have a consistent citation style.

- describe the methods you used to perform your analysis from beginning to end that narrates the analysis code.

your report should include code which:
- loads data 
- wrangles and cleans the data to the format necessary for the planned analysis
- performs a summary of the data set that is relevant for exploratory data analysis related to the planned analysis 
- creates a visualization of the dataset that is relevant for exploratory data analysis related to the planned analysis
- performs the data analysis
- creates a visualization of the analysis 
- note: all figures should have a figure number and a legend


In [60]:
### Run this cell before continuing.
import numpy as np
import pandas as pd
import altair as alt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn import set_config


# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

# Merge through shared hashedEmail
players = pd.read_csv("players.csv")
sessions = pd.read_csv("sessions.csv")
players_merged = players.merge(sessions, on="hashedEmail", how="inner")
players_clean = players_merged.drop(columns=["individualId", "organizationName"])

# Calculate each playtime duration in minutes
players_clean['start_time'] = pd.to_datetime(players_clean['start_time'], format='%d/%m/%Y %H:%M')
players_clean['end_time'] = pd.to_datetime(players_clean['end_time'], format='%d/%m/%Y %H:%M')
players_clean['duration_minutes'] = (players_clean['end_time'] - players_clean['start_time']).dt.total_seconds() / 60

# Calculate total playtime
agg = (
    players_clean.groupby("hashedEmail").agg(
        total_sessions=("hashedEmail", "count"),
        avg_session_length_minutes=("duration_minutes", "mean"),
        total_session_time_minutes=("duration_minutes", "sum")
    ).reset_index()
)

# Merge agg with players
players_total = players_clean.merge(agg, on="hashedEmail", how="inner")

# Get rid of duplicates and unneccessary data
players_total = players_total.drop(columns=["original_start_time", "original_end_time", "start_time", "end_time", "duration_minutes", "played_hours"])
players_total = players_total.drop_duplicates()


players_total

columns_to_plot = players_total.loc[:, "age" : "total_session_time_minutes"].columns.tolist()


pm_pairs = alt.Chart(players_total).mark_circle(opacity=0.2).encode(
    alt.X(alt.repeat("row"), type="quantitative"),
    alt.Y(alt.repeat("column"), type="quantitative"),
).properties(
    width=150,
    height=150
).repeat(
    column=columns_to_plot,
    row=columns_to_plot
)
pm_pairs



alt.RepeatChart(...)

## Discussion

- summarize what you found
- discuss whether this is what you expected to find?
- discuss what impact could such findings have?
- discuss what future questions could this lead to?

## References